# MedChat: Drug Interaction Chatbot Using DDInter and an LLM

This notebook is my final project implementation. The main idea is to build a conversational system that helps a user check possible drug-drug interactions. The medical interaction information comes from the DDInter CSV files in this project folder. The LLM is used for conversation style and explanation. Exact interaction severity is retrieved from the dataset, and a champion ML model is used as a fallback predictor when a pair is missing from the local database.

The notebook is organized as a full pipeline:

1. Load and clean the DDInter data.
2. Store the data in SQLite so the chatbot can query it quickly.
3. Extract medication names from user text using a drug vocabulary, synonyms, and fuzzy matching.
4. Ask the user several intake questions before making a recommendation.
5. Look up all drug pairs in the local database.
6. Use the OpenAI API to turn retrieved DDInter evidence and champion ML predictions into a natural response.
7. Evaluate the extraction and lookup logic with a few test cases.
8. Run a Gradio demo for presentation.

A `.env` file is used for the OpenAI key. The notebook only checks whether the key exists; it does not print the key.


## 0. Google Colab Setup (run this first)

This cell installs the libraries MedChat needs and detects whether the
notebook is running on Google Colab. Run it once at the start of every
Colab session (Colab resets its environment each time).


In [1]:
# --- Install project dependencies (safe to re-run) ---
# Colab already has pandas, numpy, scikit-learn, matplotlib, seaborn, joblib.
# These are the ones Colab does NOT ship by default:
%pip install -q rapidfuzz openai python-dotenv gradio

# --- Detect Colab ---
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    print('Running on Google Colab.')
except Exception:
    IN_COLAB = False
    print('Not running on Colab (local/other environment).')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.6 MB/s eta 0:00:00
Running on Google Colab.


## 1. Imports and Configuration

This cell imports the libraries used in the project and sets up the main paths. The important paths are:

- `DATA_PATH`: where the DDInter CSV files are located.
- `OUTPUT_PATH`: where generated plots and the SQLite database are saved.
- `DB_PATH`: the SQLite database file used later for interaction lookup.

This cell also loads `.env` so that `OPENAI_API_KEY` can be used without writing the key directly in the notebook. If the key is missing, the notebook still works with fallback responses, which is useful for testing the data pipeline without spending API calls.


In [3]:
from pathlib import Path
from itertools import combinations
import os
import re
import json
import sqlite3
import socket
import getpass
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

try:
    from openai import OpenAI
except Exception as exc:
    OpenAI = None
    print("OpenAI SDK is not available:", exc)

# ------------------------------------------------------------------
# Paths. On Colab we work inside /content; locally we use the CWD.
# ------------------------------------------------------------------
if 'IN_COLAB' in globals() and IN_COLAB:
    DATA_PATH = Path('/content/medchat_data')
    DATA_PATH.mkdir(parents=True, exist_ok=True)
else:
    DATA_PATH = Path.cwd()

OUTPUT_PATH = DATA_PATH / "medchat_llm_outputs"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
DB_PATH = OUTPUT_PATH / "ddinter_llm_demo.db"
# Optional pre-built DB fallback used by the flexible loader below.
SOURCE_DB_PATH = DATA_PATH / "ddinter.db"
ENV_PATH = DATA_PATH / ".env"

# ------------------------------------------------------------------
# Load the OpenAI API key WITHOUT pasting it into the notebook.
# Priority: 1) Colab Secrets  2) existing env var  3) .env file  4) prompt.
#
# COLAB SETUP (recommended): click the key icon in the left sidebar,
# add a secret named OPENAI_API_KEY, paste your key as the value, and
# enable 'Notebook access'. Then this cell picks it up automatically.
# ------------------------------------------------------------------
def _load_openai_key():
    # 1) Colab Secrets manager
    try:
        from google.colab import userdata
        key = userdata.get('OPENAI_API_KEY')
        if key:
            return key.strip(), 'Colab Secrets'
    except Exception:
        pass
    # 2) Already in the environment
    if os.getenv('OPENAI_API_KEY'):
        return os.environ['OPENAI_API_KEY'].strip(), 'environment variable'
    # 3) Local .env file
    if load_dotenv is not None and ENV_PATH.exists():
        load_dotenv(ENV_PATH)
    elif ENV_PATH.exists():
        for line in ENV_PATH.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line:
                continue
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
    if os.getenv('OPENAI_API_KEY'):
        return os.environ['OPENAI_API_KEY'].strip(), '.env file'
    # 4) Prompt (hidden input) as a last resort
    try:
        key = getpass.getpass('Paste your OpenAI API key (input hidden): ').strip()
        if key:
            return key, 'manual entry'
    except Exception:
        pass
    return None, 'not found'

OPENAI_API_KEY, KEY_SOURCE = _load_openai_key()
if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
USE_OPENAI = bool(OPENAI_API_KEY and OpenAI is not None)

print("Data path:", DATA_PATH)
print("Output path:", OUTPUT_PATH)
print("OpenAI SDK available:", OpenAI is not None)
print("OpenAI key source:", KEY_SOURCE)
print("OpenAI key loaded:", bool(OPENAI_API_KEY))
print("OpenAI enabled:", USE_OPENAI)
print("Model:", OPENAI_MODEL)


Data path: /content/medchat_data
Output path: /content/medchat_data/medchat_llm_outputs
OpenAI SDK available: True
OpenAI key source: Colab Secrets
OpenAI key loaded: True
OpenAI enabled: True
Model: gpt-4.1-mini


## 2. Load and Prepare the DDInter Dataset

The DDInter files are split across several CSV files. In this section, I load every file whose name starts with `ddinter_downloads_code_`, combine them into one dataframe, and remove duplicate rows.

The important columns are:

- `Drug_A`: first drug in the pair.
- `Drug_B`: second drug in the pair.
- `Level`: interaction severity, such as `Major`, `Moderate`, `Minor`, or `Unknown`.

After cleaning, the dataframe is saved into SQLite. This makes the chatbot easier to design because later functions can query one database table instead of scanning CSV files every time.


In [4]:
# ------------------------------------------------------------------
# Get the 8 DDInter CSVs into DATA_PATH.
#
# On Colab, pick ONE of the two options below by leaving it uncommented.
#   Option A (simplest): upload the CSVs from your computer.
#   Option B: mount Google Drive and copy them from a Drive folder.
# If the CSVs are already in DATA_PATH (local run), both options are skipped.
# ------------------------------------------------------------------
REQUIRED_COLUMNS = {"Drug_A", "Drug_B", "Level"}
OPTIONAL_COLUMNS = ["DDInterID_A", "DDInterID_B"]

existing_csvs = sorted(DATA_PATH.glob("ddinter_downloads_code_*.csv"))

if ('IN_COLAB' in globals() and IN_COLAB) and not existing_csvs:
    # ---- Option A: upload from your computer ----
    from google.colab import files
    print("Select your 8 ddinter_downloads_code_*.csv files...")
    uploaded = files.upload()
    for name, content in uploaded.items():
        (DATA_PATH / name).write_bytes(content)
    print(f"Saved {len(uploaded)} file(s) to {DATA_PATH}")

    # ---- Option B: mount Google Drive (uncomment to use instead) ----
    # from google.colab import drive
    # drive.mount('/content/drive')
    # drive_folder = Path('/content/drive/MyDrive/medchat_data')
    # for f in drive_folder.glob('ddinter_downloads_code_*.csv'):
    #     (DATA_PATH / f.name).write_bytes(f.read_bytes())


def load_ddinter_from_csvs(data_path):
    csv_files = sorted(data_path.glob("ddinter_downloads_code_*.csv"))
    if not csv_files:
        csv_files = sorted(data_path.glob("*.csv"))
    if not csv_files:
        return None, []
    frames, skipped = [], []
    print(f"Found {len(csv_files)} CSV file(s) in {data_path}. Checking columns...")
    for file in csv_files:
        try:
            temp = pd.read_csv(file, low_memory=False)
            temp.columns = [str(c).strip() for c in temp.columns]
            if not REQUIRED_COLUMNS.issubset(set(temp.columns)):
                skipped.append(file.name)
                continue
            keep = [c for c in OPTIONAL_COLUMNS if c in temp.columns] + ["Drug_A", "Drug_B", "Level"]
            temp = temp[keep].copy()
            temp["Drug_A"] = temp["Drug_A"].astype(str).str.strip()
            temp["Drug_B"] = temp["Drug_B"].astype(str).str.strip()
            temp["Level"] = temp["Level"].astype(str).str.strip().replace({"": "Unknown", "nan": "Unknown"})
            temp = temp.dropna(subset=["Drug_A", "Drug_B", "Level"])
            temp = temp[(temp["Drug_A"] != "") & (temp["Drug_B"] != "")]
            frames.append(temp)
            print(f"Loaded {file.name}: {len(temp):,} usable rows")
        except Exception as e:
            skipped.append(f"{file.name} ({e})")
    if skipped:
        print("\nSkipped non-DDInter or unreadable CSVs:")
        for item in skipped:
            print(" -", item)
    if not frames:
        return None, skipped
    combined = pd.concat(frames, ignore_index=True).drop_duplicates().reset_index(drop=True)
    return combined, skipped


def load_ddinter_from_existing_db(source_db_path):
    if not source_db_path.exists():
        return None
    print(f"\nNo usable CSVs found. Trying existing SQLite database: {source_db_path}")
    conn = sqlite3.connect(source_db_path)
    try:
        df = pd.read_sql_query("SELECT * FROM interactions", conn)
    finally:
        conn.close()
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f"Existing DB missing columns: {missing}")
    keep = [c for c in OPTIONAL_COLUMNS if c in df.columns] + ["Drug_A", "Drug_B", "Level"]
    df = df[keep].copy()
    df["Drug_A"] = df["Drug_A"].astype(str).str.strip()
    df["Drug_B"] = df["Drug_B"].astype(str).str.strip()
    df["Level"] = df["Level"].astype(str).str.strip().replace({"": "Unknown", "nan": "Unknown"})
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Loaded {len(df):,} interaction rows from existing database.")
    return df


interactions, skipped_files = load_ddinter_from_csvs(DATA_PATH)
if interactions is None:
    interactions = load_ddinter_from_existing_db(SOURCE_DB_PATH)
if interactions is None or interactions.empty:
    raise FileNotFoundError(
        "No usable DDInter data found. Upload the ddinter_downloads_code_*.csv files "
        f"to {DATA_PATH}, or provide {SOURCE_DB_PATH} with an interactions table."
    )

missing = REQUIRED_COLUMNS - set(interactions.columns)
if missing:
    raise ValueError(f"Missing expected columns after loading: {missing}")

conn = sqlite3.connect(DB_PATH)
interactions.to_sql("interactions", conn, if_exists="replace", index=False)
conn.close()

print(f"\nCombined unique interactions: {len(interactions):,}")
print(f"Saved SQLite database for this notebook: {DB_PATH}")
print(interactions.head())


Select your 8 ddinter_downloads_code_*.csv files...


Saving ddinter_downloads_code_A.csv to ddinter_downloads_code_A.csv
Saving ddinter_downloads_code_B.csv to ddinter_downloads_code_B.csv
Saving ddinter_downloads_code_D.csv to ddinter_downloads_code_D.csv
Saving ddinter_downloads_code_H.csv to ddinter_downloads_code_H.csv
Saving ddinter_downloads_code_L.csv to ddinter_downloads_code_L.csv
Saving ddinter_downloads_code_P.csv to ddinter_downloads_code_P.csv
Saving ddinter_downloads_code_R.csv to ddinter_downloads_code_R.csv
Saving ddinter_downloads_code_V.csv to ddinter_downloads_code_V.csv
Saved 8 file(s) to /content/medchat_data
Found 8 CSV file(s) in /content/medchat_data. Checking columns...
Loaded ddinter_downloads_code_A.csv: 56,367 usable rows
Loaded ddinter_downloads_code_B.csv: 15,140 usable rows
Loaded ddinter_downloads_code_D.csv: 25,681 usable rows
Loaded ddinter_downloads_code_H.csv: 11,727 usable rows
Loaded ddinter_downloads_code_L.csv: 65,389 usable rows
Loaded ddinter_downloads_code_P.csv: 5,492 usable rows
Loaded ddinter

In [5]:
severity_counts = interactions["Level"].value_counts()
print(severity_counts)

plt.figure(figsize=(7, 4))
sns.barplot(x=severity_counts.index, y=severity_counts.values)
plt.title("DDInter Severity Distribution")
plt.xlabel("Severity")
plt.ylabel("Number of Interaction Records")
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "severity_distribution.png", dpi=150)
plt.close()


Level
Moderate    96675
Unknown     29813
Major       26914
Minor        6833
Name: count, dtype: int64


## 3. Drug Name Resolution

User input is usually not written in the same format as the dataset. For example, a user may type `Advil`, but the dataset stores the drug as `Ibuprofen`. This section handles that problem.

The code builds a drug vocabulary from all `Drug_A` and `Drug_B` values in DDInter. Then it uses three matching steps:

1. Exact matching against known dataset names.
2. Synonym matching for common brand names and misspellings.
3. Fuzzy matching for small spelling mistakes.

I also use a `FALSE_POSITIVES` set so common words like `taking`, `disease`, or `condition` are not accidentally treated as medication names.


In [6]:
SYNONYMS = {
    "advil": "Ibuprofen",
    "motrin": "Ibuprofen",
    "tylenol": "Acetaminophen",
    "paracetamol": "Acetaminophen",
    "aspirin": "Acetylsalicylic acid",
    "bayer": "Acetylsalicylic acid",
    "coumadin": "Warfarin",
    "glucophage": "Metformin",
    "prilosec": "Omeprazole",
    "zoloft": "Sertraline",
    "prozac": "Fluoxetine",
    "xanax": "Alprazolam",
    "lipitor": "Atorvastatin",
    "norvasc": "Amlodipine",
    "lasix": "Furosemide",
    "plavix": "Clopidogrel",
    "crestor": "Rosuvastatin",
    "toprol": "Metoprolol",
    "warafin": "Warfarin",
    "warferin": "Warfarin",
    "ibuprofin": "Ibuprofen",
    "metforim": "Metformin",
    "metformine": "Metformin",
    "amoxicilin": "Amoxicillin",
    "potassium": "Potassium chloride",
    "potassium supplement": "Potassium chloride",
}

FALSE_POSITIVES = {
    "take", "taking", "drug", "drugs", "medicine", "medication", "medications",
    "pill", "pills", "tablet", "tablets", "capsule", "dose", "daily", "day",
    "week", "month", "condition", "conditions", "kidney", "liver", "heart",
    "disease", "pressure", "blood", "pregnant", "pregnancy", "asthma", "allergy",
    "allergies", "allergic", "none", "yes", "no", "what", "about", "with",
    "and", "or", "also", "remove", "stopped", "started"
}

all_drugs = pd.concat([interactions["Drug_A"], interactions["Drug_B"]]).dropna().astype(str).drop_duplicates().tolist()
exact_drugs = {drug.lower(): drug for drug in all_drugs}
print("Unique DDInter drug names:", len(all_drugs))


def fuzzy_match_drug(name, threshold=86):
    cleaned = str(name).lower().strip()
    if len(cleaned) <= 3 or cleaned in FALSE_POSITIVES or cleaned.isdigit():
        return None
    if cleaned in SYNONYMS:
        return SYNONYMS[cleaned]
    if cleaned in exact_drugs:
        return exact_drugs[cleaned]
    result = process.extractOne(name, all_drugs, scorer=fuzz.token_sort_ratio, score_cutoff=threshold)
    return result[0] if result else None


def extract_drugs(text):
    text_lower = text.lower()
    candidates = []

    # Multi-word and exact drug names found directly in the input.
    for key, value in exact_drugs.items():
        if len(key) > 4 and re.search(rf"\b{re.escape(key)}\b", text_lower):
            candidates.append(value)

    # Synonyms and typo dictionary.
    for token in re.findall(r"\b[a-zA-Z][a-zA-Z-]+\b", text_lower):
        if token in SYNONYMS:
            candidates.append(SYNONYMS[token])

    # Conservative fuzzy matching for remaining tokens.
    for token in re.findall(r"\b[a-zA-Z][a-zA-Z-]+\b", text_lower):
        if token in FALSE_POSITIVES or token in SYNONYMS:
            continue
        if len(token) >= 6:
            match = fuzzy_match_drug(token)
            if match:
                candidates.append(match)

    resolved = []
    seen = set()
    for drug in candidates:
        if drug.lower() not in seen:
            resolved.append(drug)
            seen.add(drug.lower())
    return resolved

for query in [
    "I take warfarin and ibuprofen",
    "Can I take Advil with Coumadin?",
    "I use metforim and prilosec",
    "I have kidney disease",
]:
    print(query, "->", extract_drugs(query))


Unique DDInter drug names: 1939
I take warfarin and ibuprofen -> ['Warfarin', 'Ibuprofen']
Can I take Advil with Coumadin? -> ['Ibuprofen', 'Warfarin']
I use metforim and prilosec -> ['Metformin', 'Omeprazole']
I have kidney disease -> []


## 4. Interaction Lookup and Recommendation Evidence

This section contains the functions that connect the chatbot to the actual dataset.

`find_interaction(drug1, drug2)` searches the SQLite table for a pair in either order. This matters because the user might say `warfarin and ibuprofen`, while the database could store the pair as `Ibuprofen` and `Warfarin`.

`check_interaction_pairs(drugs)` checks every possible pair in the user's medication list. The result is a structured evidence list. This evidence is what the LLM receives later, so the generated answer stays grounded in DDInter instead of relying on model memory.


In [7]:
def find_interaction(drug1, drug2):
    conn = sqlite3.connect(DB_PATH)
    query = """
        SELECT Drug_A, Drug_B, Level
        FROM interactions
        WHERE (LOWER(Drug_A) = LOWER(?) AND LOWER(Drug_B) = LOWER(?))
           OR (LOWER(Drug_A) = LOWER(?) AND LOWER(Drug_B) = LOWER(?))
        LIMIT 1
    """
    result = pd.read_sql_query(query, conn, params=[drug1, drug2, drug2, drug1])
    conn.close()
    if result.empty:
        return None
    return result.iloc[0].to_dict()


def severity_recommendation(level):
    level = str(level).lower()
    if level == "major":
        return "High concern: do not combine unless a clinician specifically instructed it. Contact a doctor or pharmacist."
    if level == "moderate":
        return "Use caution: ask a clinician whether monitoring, timing, or dose changes are needed."
    if level == "minor":
        return "Lower concern: monitor symptoms and confirm with a pharmacist if unsure."
    return "Unclear severity: do not assume it is safe; ask a pharmacist."


def check_interaction_pairs(drugs):
    evidence = []
    for drug1, drug2 in combinations(drugs, 2):
        row = find_interaction(drug1, drug2)
        if row:
            evidence.append({
                "drug1": drug1,
                "drug2": drug2,
                "found": True,
                "level": row["Level"],
                "recommendation": severity_recommendation(row["Level"]),
            })
        else:
            ml_prediction = None
            if "predict_interaction_severity" in globals():
                ml_prediction = predict_interaction_severity(drug1, drug2)

            if ml_prediction:
                evidence.append({
                    "drug1": drug1,
                    "drug2": drug2,
                    "found": False,
                    "level": "Not found",
                    "recommendation": "No exact DDInter record found. The champion ML model predicts "
                    f"{ml_prediction['level']} severity ({ml_prediction['source']}). "
                    f"{ml_prediction['recommendation']}",
                    "ml_prediction": ml_prediction["level"],
                    "ml_source": ml_prediction["source"],
                })
            else:
                evidence.append({
                    "drug1": drug1,
                    "drug2": drug2,
                    "found": False,
                    "level": "Not found",
                    "recommendation": "No DDInter record found. This does not prove the combination is safe.",
                })
    return evidence

print(check_interaction_pairs(["Warfarin", "Ibuprofen"]))


[{'drug1': 'Warfarin', 'drug2': 'Ibuprofen', 'found': True, 'level': 'Major', 'recommendation': 'High concern: do not combine unless a clinician specifically instructed it. Contact a doctor or pharmacist.'}]


## 5A. Clustering Analysis

This section adds the unsupervised learning part required by the rubric. I use TF-IDF text features from `Drug_A`, `Drug_B`, and `Level`, reduce the feature space with TruncatedSVD, and apply KMeans clustering. The purpose is not to make the final medical decision, but to discover groups of similar interaction records and understand the severity patterns inside those groups.

In [8]:
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import TfidfVectorizer

cluster_data = interactions[["Drug_A", "Drug_B", "Level"]].dropna().copy()
cluster_sample_size = min(20000, len(cluster_data))
cluster_sample = cluster_data.sample(n=cluster_sample_size, random_state=42).reset_index(drop=True)
cluster_sample["cluster_text"] = (
    cluster_sample["Drug_A"].astype(str) + " " +
    cluster_sample["Drug_B"].astype(str) + " " +
    cluster_sample["Level"].astype(str)
)

cluster_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), lowercase=True)
X_cluster = cluster_vectorizer.fit_transform(cluster_sample["cluster_text"])

svd = TruncatedSVD(n_components=50, random_state=42)
X_cluster_reduced = svd.fit_transform(X_cluster)

candidate_ks = [3, 4, 5, 6, 7]
silhouette_rows = []
for k in candidate_ks:
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_cluster_reduced)
    score = silhouette_score(
        X_cluster_reduced,
        labels,
        sample_size=min(5000, len(labels)),
        random_state=42,
    )
    silhouette_rows.append({"k": k, "silhouette_score": score})

silhouette_df = pd.DataFrame(silhouette_rows)
best_k = int(silhouette_df.sort_values("silhouette_score", ascending=False).iloc[0]["k"])

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_sample["cluster"] = kmeans.fit_predict(X_cluster_reduced)

cluster_summary = (
    cluster_sample.groupby("cluster")
    .agg(
        records=("cluster", "size"),
        dominant_severity=("Level", lambda x: x.value_counts().idxmax()),
        dominant_severity_count=("Level", lambda x: x.value_counts().max()),
    )
    .reset_index()
)
cluster_summary["dominant_severity_pct"] = (
    cluster_summary["dominant_severity_count"] / cluster_summary["records"]
).round(3)

print(f"Clustering sample size: {cluster_sample_size:,} DDInter interaction records")
print("Text features: TF-IDF on Drug_A + Drug_B + Level")
print("Dimensionality reduction: TruncatedSVD to 50 components")
print(f"Best k by silhouette score: {best_k}")
print("\nSilhouette scores:")
display(silhouette_df)
print("Cluster summary:")
display(cluster_summary)

severity_by_cluster = pd.crosstab(cluster_sample["cluster"], cluster_sample["Level"], normalize="index")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.lineplot(data=silhouette_df, x="k", y="silhouette_score", marker="o", ax=axes[0])
axes[0].set_title("KMeans Silhouette Score by k")
axes[0].set_xlabel("Number of clusters")
axes[0].set_ylabel("Silhouette score")
severity_by_cluster.plot(kind="bar", stacked=True, ax=axes[1], colormap="viridis")
axes[1].set_title("Severity Mix Inside Each Cluster")
axes[1].set_xlabel("Cluster")
axes[1].set_ylabel("Proportion of records")
axes[1].legend(title="Severity", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

Clustering sample size: 20,000 DDInter interaction records
Text features: TF-IDF on Drug_A + Drug_B + Level
Dimensionality reduction: TruncatedSVD to 50 components
Best k by silhouette score: 6

Silhouette scores:


,k,silhouette_score
0,3,0.223583
1,4,0.185828
2,5,0.242433
3,6,0.264699
4,7,0.242969


Cluster summary:


,cluster,records,dominant_severity,dominant_severity_count,dominant_severity_pct
0,0,11521,Moderate,11521,1.000
1,1,413,Moderate,209,0.506
2,2,841,Minor,841,1.000
3,3,3129,Major,3129,1.000
4,4,3443,Unknown,3443,1.000
5,5,653,Moderate,331,0.507



## 5. ML Severity Prediction and Champion Model Selection

The exact SQLite lookup is still the most reliable source when a drug pair exists in DDInter. However, the local DDInter files may not contain every possible pair. To make the project more complete as a machine learning project, this section trains several supervised models to predict the interaction severity class.

The feature is a text representation of each drug pair, and the target label is DDInter `Level`.

Models compared:

- Linear SVM (`LinearSVC`)
- Logistic Regression
- Multinomial Naive Bayes
- SGD linear classifier

I evaluate all models on the same train/test split and choose the champion model using weighted F1 score. The chatbot uses this champion model only when an exact DDInter pair is missing. In the chatbot response, this is clearly described as an ML estimate rather than confirmed DDInter evidence.


In [9]:

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, f1_score
import joblib

ml_data = interactions[["Drug_A", "Drug_B", "Level"]].dropna().copy()
ml_data["pair_text"] = (
    ml_data["Drug_A"].astype(str) + " " +
    ml_data["Drug_B"].astype(str) + " " +
    ml_data["Drug_B"].astype(str) + " " +
    ml_data["Drug_A"].astype(str)
)

# A sample keeps training fast enough for a live notebook demo while still using real DDInter data.
train_sample_size = min(60000, len(ml_data))
ml_sample = ml_data.sample(n=train_sample_size, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    ml_sample["pair_text"],
    ml_sample["Level"],
    test_size=0.2,
    random_state=42,
    stratify=ml_sample["Level"]
)

model_candidates = {
    "Linear SVM": LinearSVC(class_weight="balanced", random_state=42),
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="lbfgs",
        random_state=42
    ),
    "Multinomial Naive Bayes": MultinomialNB(),
    "SGD Classifier": SGDClassifier(
        loss="modified_huber",
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),
}

model_results = []
trained_models = {}

for model_name, estimator in model_candidates.items():
    print(f"Training {model_name}...")
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), lowercase=True)),
        ("model", estimator)
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    weighted_f1 = f1_score(y_test, predictions, average="weighted")
    macro_f1 = f1_score(y_test, predictions, average="macro")

    model_results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Weighted F1": weighted_f1,
        "Macro F1": macro_f1,
    })
    trained_models[model_name] = pipeline

model_results_df = pd.DataFrame(model_results).sort_values("Weighted F1", ascending=False).reset_index(drop=True)
print("\nModel comparison:")
print(model_results_df.to_string(index=False, formatters={
    "Accuracy": "{:.3f}".format,
    "Weighted F1": "{:.3f}".format,
    "Macro F1": "{:.3f}".format,
}))

champion_model_name = model_results_df.loc[0, "Model"]
champion_model = trained_models[champion_model_name]
champion_predictions = champion_model.predict(X_test)

print(f"\nChampion model: {champion_model_name}")
print(classification_report(y_test, champion_predictions))

plt.figure(figsize=(8, 4))
sns.barplot(data=model_results_df, x="Weighted F1", y="Model")
plt.title("ML Model Comparison for Severity Prediction")
plt.xlim(0, 1)
plt.tight_layout()
plt.savefig(OUTPUT_PATH / "ml_model_comparison.png", dpi=150)
plt.close()

MODEL_PATH = OUTPUT_PATH / "champion_severity_model.joblib"
joblib.dump({
    "model_name": champion_model_name,
    "model": champion_model,
    "results": model_results_df,
}, MODEL_PATH)
print("Saved champion model to:", MODEL_PATH)


def predict_interaction_severity(drug1, drug2):
    pair_text = f"{drug1} {drug2} {drug2} {drug1}"
    prediction = champion_model.predict([pair_text])[0]
    confidence_note = f"{champion_model_name} estimate"

    estimator = champion_model.named_steps["model"]

    if hasattr(estimator, "predict_proba"):
        probabilities = champion_model.predict_proba([pair_text])[0]
        confidence = float(np.max(probabilities))
        if confidence < 0.55:
            confidence_note = f"low-confidence {champion_model_name} estimate"
        else:
            confidence_note = f"{champion_model_name} estimate, confidence {confidence:.2f}"
    elif hasattr(estimator, "decision_function"):
        scores = champion_model.decision_function([pair_text])
        if getattr(scores, "ndim", 1) == 2 and scores.shape[1] > 1:
            sorted_scores = np.sort(scores[0])
            margin = float(sorted_scores[-1] - sorted_scores[-2])
            if margin < 0.5:
                confidence_note = f"low-confidence {champion_model_name} estimate"

    return {
        "level": prediction,
        "recommendation": severity_recommendation(prediction),
        "source": confidence_note,
        "model_name": champion_model_name,
    }


Training Linear SVM...
Training Logistic Regression...
Training Multinomial Naive Bayes...
Training SGD Classifier...

Model comparison:
                  Model Accuracy Weighted F1 Macro F1
             Linear SVM    0.815       0.819    0.746
         SGD Classifier    0.813       0.816    0.744
    Logistic Regression    0.780       0.789    0.717
Multinomial Naive Bayes    0.804       0.788    0.665

Champion model: Linear SVM
              precision    recall  f1-score   support

       Major       0.69      0.77      0.73      2028
       Minor       0.49      0.66      0.56       509
    Moderate       0.91      0.81      0.86      7237
     Unknown       0.77      0.92      0.84      2226

    accuracy                           0.82     12000
   macro avg       0.72      0.79      0.75     12000
weighted avg       0.83      0.82      0.82     12000

Saved champion model to: /content/medchat_data/medchat_llm_outputs/champion_severity_model.joblib


## 6. Guided Conversation State

A normal chatbot response is not enough for this project because the system should feel like a multi-turn assistant. To do that, I keep a session dictionary that stores what the user has already told the chatbot.

The state includes:

- medication names
- dose or timing notes
- allergies
- medical conditions
- age group
- the user's main concern
- the last interaction evidence retrieved from DDInter
- the current conversation stage

The `stage` value controls what the assistant asks next. This makes the chatbot follow an intake flow instead of immediately jumping to a recommendation.


In [10]:

CONDITION_KEYWORDS = [
    "kidney disease", "kidney problems", "liver disease", "liver problems",
    "pregnant", "pregnancy", "asthma", "diabetes", "hypertension",
    "high blood pressure", "heart disease", "ulcer", "bleeding disorder",
    "stroke", "blood clot", "depression", "anxiety"
]

EMERGENCY_PHRASES = [
    "chest pain", "can't breathe", "cannot breathe", "shortness of breath",
    "severe bleeding", "black stool", "blood in vomit", "fainting",
    "seizure", "overdose", "suicidal", "throat closing"
]

AGE_GROUPS = ["child", "teen", "adult", "older adult", "senior"]


def new_session():
    return {
        "stage": "medications",
        "drugs": [],
        "dosing_notes": [],
        "allergies": [],
        "conditions": [],
        "age_group": None,
        "main_concern": None,
        "confirmed": False,
        "last_evidence": [],
        "last_pair": None,
        "turns": 0,
    }


def normalize_session(state):
    base = new_session()
    if isinstance(state, dict):
        base.update(state)
        for key in ["drugs", "dosing_notes", "allergies", "conditions", "last_evidence"]:
            if base.get(key) is None:
                base[key] = []
    return base


def add_unique(target, values):
    added = []
    existing = {str(x).lower() for x in target}
    for value in values:
        if value and str(value).lower() not in existing:
            target.append(value)
            existing.add(str(value).lower())
            added.append(value)
    return added


def detect_conditions(text):
    lower = text.lower()
    return [c for c in CONDITION_KEYWORDS if c in lower]


def detect_age_group(text):
    lower = text.lower()
    for group in AGE_GROUPS:
        if group in lower:
            return group
    match = re.search(r'\b(\d{1,3})\s*(years old|year old|yo)?\b', lower)
    if match:
        age = int(match.group(1))
        if age < 13:
            return "child"
        if age < 18:
            return "teen"
        if age >= 65:
            return "older adult"
        return "adult"
    return None


def extract_dosing_notes(text):
    notes = []
    patterns = [
        r'\b\d+(?:\.\d+)?\s*(?:mg|mcg|g|ml|units?)\b',
        r'\b(?:once|twice|three times|\d+ times)\s+(?:a|per)\s+day\b',
        r'\b(?:morning|night|evening|daily|weekly|as needed|prn)\b',
    ]
    for pattern in patterns:
        notes.extend(re.findall(pattern, text, flags=re.I))
    return [str(note).strip() for note in notes]


def extract_allergies(text):
    lower = text.lower()
    if not any(word in lower for word in ["allergic", "allergy", "allergies"]):
        return []
    drugs = extract_drugs(text)
    if drugs:
        return drugs
    if "none" in lower or "no" == lower.strip():
        return []
    cleaned = re.sub(r".*allerg(?:ic|y|ies)\s*(to)?", "", text, flags=re.I).strip()
    parts = [p.strip().title() for p in re.split(r",| and ", cleaned) if p.strip()]
    return parts[:4]


def is_negative_answer(text):
    return text.lower().strip() in {"no", "none", "nope", "not sure", "unknown", "i don't know", "dont know", "skip"}


def is_affirmative_answer(text):
    return text.lower().strip() in {"yes", "yeah", "yep", "sure", "ok", "okay", "correct", "looks good", "confirm", "confirmed"}


def detect_emergency(text):
    return any(phrase in text.lower() for phrase in EMERGENCY_PHRASES)


def next_question(state):
    if state["stage"] == "medications":
        return "Please list the medications, supplements, or over-the-counter drugs you take. Brand names are okay."
    if state["stage"] == "dosing":
        return "Do you know the dose or timing for any of them? For example, `warfarin 5 mg at night`. You can also say `skip`."
    if state["stage"] == "allergies":
        return "Do you have any medication allergies? You can say `none`, or list them."
    if state["stage"] == "conditions":
        return "Do you have important health factors like kidney disease, liver disease, pregnancy, asthma, ulcers, or bleeding problems?"
    if state["stage"] == "age":
        return "What age group should I consider: child, teen, adult, or older adult? You can also type your age."
    if state["stage"] == "concern":
        return "What is your main concern today: safety, side effects, what to watch for, or whether to ask a pharmacist?"
    if state["stage"] == "confirm":
        return "Before I check the dataset, please confirm: is this profile correct? Reply `yes` to continue or add corrections."
    if state["stage"] == "ready":
        return "You can ask a follow-up like `what could happen?`, `what about aspirin?`, or `remove ibuprofen`."
    return "How can I help with your medication interaction question?"


def profile_text(state):
    last_pair = state.get("last_pair")
    last_pair_text = " + ".join(last_pair) if last_pair else "None yet"
    return (
        f"Medications: {', '.join(state['drugs']) if state['drugs'] else 'None yet'}\n"
        f"Dose/timing notes: {', '.join(state['dosing_notes']) if state['dosing_notes'] else 'None recorded'}\n"
        f"Allergies: {', '.join(state['allergies']) if state['allergies'] else 'None recorded'}\n"
        f"Conditions: {', '.join(state['conditions']) if state['conditions'] else 'None recorded'}\n"
        f"Age group: {state['age_group'] or 'Not recorded'}\n"
        f"Main concern: {state['main_concern'] or 'Not recorded'}\n"
        f"Last checked pair: {last_pair_text}\n"
        f"Conversation stage: {state['stage']}"
    )


## 7. OpenAI LLM Response Layer

The project uses the OpenAI API only after the deterministic code has extracted entities, retrieved DDInter evidence, and generated any needed champion ML prediction. This is important because the LLM should explain the evidence, not invent the evidence.

The `SYSTEM_INSTRUCTIONS` variable tells the model to:

- stay grounded in the provided DDInter evidence
- avoid inventing medical facts
- ask short follow-up questions during intake
- recommend a doctor or pharmacist for major interactions
- clearly state that this is educational and not medical advice

If the API key is not available, `call_openai()` returns a fallback response. This makes the notebook runnable even without API access.


In [11]:
client = OpenAI(api_key=OPENAI_API_KEY) if USE_OPENAI else None

SYSTEM_INSTRUCTIONS = """
You are MedChat, a final-project drug interaction assistant.
Your job is to make the conversation natural, calm, and useful while staying strictly grounded in the provided dataset evidence.

Rules:
- Do not invent interactions, severity levels, side effects, or medical facts that are not in the evidence.
- If the evidence says no DDInter record was found, say that this does not prove safety.
- Ask one short follow-up question at a time during intake.
- Be clear that this is educational and not medical advice.
- For Major interactions, recommend contacting a doctor or pharmacist before combining.
- For emergency symptoms, advise urgent care.
- Keep answers concise enough for a live demo.
"""


def call_openai(task, state, evidence=None, fallback=None):
    if not USE_OPENAI:
        return fallback or task

    payload = {
        "task": task,
        "session_state": state,
        "dataset_evidence": evidence or [],
    }

    try:
        response = client.responses.create(
            model=OPENAI_MODEL,
            instructions=SYSTEM_INSTRUCTIONS,
            input=[
                {
                    "role": "user",
                    "content": json.dumps(payload, indent=2)
                }
            ],
        )
        return response.output_text.strip()
    except Exception as exc:
        print("OpenAI call failed; using fallback:", exc)
        return fallback or task


def fallback_recommendation(state, evidence):
    if not evidence:
        return next_question(state)

    lines = ["Here is what I found in the DDInter dataset:"]
    major = False
    for item in evidence:
        lines.append(f"- {item['drug1']} + {item['drug2']}: {item['level']}")
        lines.append(f"  Recommendation: {item['recommendation']}")
        if str(item["level"]).lower() == "major":
            major = True

    if state["allergies"]:
        lines.append(f"Allergy note: you reported {', '.join(state['allergies'])}.")
    if state["conditions"]:
        lines.append(f"Health-factor note: you reported {', '.join(state['conditions'])}.")
    if major:
        lines.append("Important: at least one Major interaction was found. Please contact a doctor or pharmacist before combining these.")

    lines.append("Educational use only; confirm decisions with a healthcare professional.")
    return "\n".join(lines)


## 7B. openFDA Drug-Label Lookup (side effects & warnings)

DDInter tells us *whether two drugs interact and how severely*. To go beyond
severity, this layer queries the public **openFDA drug-label API** for a single
drug's common side effects, boxed warnings, and cautions. Results are cached and
fall back to a small built-in dictionary when the API is unavailable, so the demo
still works offline. As with everything else, the OpenAI layer only *rewords* this
retrieved evidence -- it never invents side effects.


In [12]:
import requests
from functools import lru_cache


def normalize_key(name):
    return str(name or "").strip().lower()


# Map canonical DDInter names to the term openFDA indexes best.
OPENFDA_SEARCH_ALIASES = {
    "Acetylsalicylic acid": "aspirin",
    "Salbutamol": "albuterol",
}

# Offline fallback so the feature still demos without network access.
COMMON_SIDE_EFFECT_FALLBACKS = {
    "sertraline": "Common side effects may include nausea, diarrhea, dry mouth, sweating, sleepiness or trouble sleeping, dizziness, tremor, and sexual side effects.",
    "ibuprofen": "Common side effects may include stomach upset, nausea, heartburn, dizziness, and stomach irritation.",
    "warfarin": "Common or important side effects may include easy bruising, bleeding, nausea, and stomach discomfort.",
    "metformin": "Most common side effects are diarrhea, nausea or vomiting, gas, weakness, indigestion, stomach discomfort, and headache.",
    "acetaminophen": "Common side effects are usually limited when used correctly, but high doses can cause serious liver injury.",
    "acetylsalicylic acid": "Common side effects may include stomach upset, heartburn, and an increased risk of bleeding or bruising.",
}


def clean_label_text(text_list, max_chars=1400):
    if not text_list:
        return None
    text = " ".join(str(t) for t in text_list) if isinstance(text_list, list) else str(text_list)
    text = re.sub(r"\s+", " ", text.replace("\n", " ")).strip()
    return (text[:max_chars] + "...") if len(text) > max_chars else text


def _list_to_text(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return " ".join(str(v) for v in value).lower()
    return str(value).lower()


def _score_openfda_result(label, drug_name):
    drug = drug_name.lower()
    openfda = label.get("openfda", {})
    generic = _list_to_text(openfda.get("generic_name"))
    brand = _list_to_text(openfda.get("brand_name"))
    substance = _list_to_text(openfda.get("substance_name"))
    score = 0
    if drug in substance: score += 6
    if drug in generic: score += 5
    if drug in brand: score += 4
    if label.get("adverse_reactions"): score += 3
    if label.get("warnings"): score += 2
    if label.get("boxed_warning"): score += 2
    if label.get("drug_interactions"): score += 2
    for token in (" and ", "/", ","):
        if token in generic: score -= 4
    return score


@lru_cache(maxsize=256)
def get_openfda_label(drug_name):
    """Return a dict of the most relevant openFDA label fields, or None."""
    search_name = OPENFDA_SEARCH_ALIASES.get(drug_name, drug_name)
    query = (
        f'openfda.generic_name:"{search_name}" '
        f'OR openfda.brand_name:"{search_name}" '
        f'OR openfda.substance_name:"{search_name}"'
    )
    try:
        resp = requests.get("https://api.fda.gov/drug/label.json",
                            params={"search": query, "limit": 25}, timeout=15)
        if resp.status_code != 200:
            return None
        results = resp.json().get("results", [])
        if not results:
            return None
        label = sorted(results, key=lambda l: _score_openfda_result(l, search_name), reverse=True)[0]
        openfda = label.get("openfda", {})
        return {
            "drug": drug_name,
            "search_name": search_name,
            "brand_name": openfda.get("brand_name"),
            "generic_name": openfda.get("generic_name"),
            "adverse_reactions": clean_label_text(label.get("adverse_reactions")),
            "warnings": clean_label_text(label.get("warnings")),
            "boxed_warning": clean_label_text(label.get("boxed_warning")),
            "contraindications": clean_label_text(label.get("contraindications")),
            "drug_interactions": clean_label_text(label.get("drug_interactions")),
        }
    except Exception as exc:
        print(f"openFDA lookup error for {drug_name}: {exc}")
        return None


def _clean_side_effect_sentence(text):
    if not text:
        return None
    replacements = {
        "adverse reactions": "side effects", "Adverse reactions": "side effects",
        "nausea/vomiting": "nausea or vomiting", "flatulence": "gas",
        "asthenia": "weakness", "abdominal discomfort": "stomach discomfort",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    for clutter in ("(>5.0%)", "(>5%)", "( 6.1 )", "(6.1)", "( 6 )", "(6)",
                    "[ see Warnings and Precautions (5.1) ]", "[see Warnings and Precautions (5.1)]"):
        text = text.replace(clutter, "")
    return re.sub(r"\s+", " ", text).strip()


def extract_short_side_effects(text, drug_name=None):
    drug_key = normalize_key(drug_name) if drug_name else None
    if drug_key in COMMON_SIDE_EFFECT_FALLBACKS:
        return COMMON_SIDE_EFFECT_FALLBACKS[drug_key]
    if not text:
        return None
    text = re.sub(r"\s+", " ", text).strip()
    for phrase in ("To report SUSPECTED ADVERSE REACTIONS", "To report suspected adverse reactions",
                   "6.1 Clinical", "Clinical Studies Experience", "Clinical Trials Experience",
                   "Because clinical trials are conducted", "During its premarketing assessment"):
        idx = text.find(phrase)
        if idx != -1:
            text = text[:idx]
    lower = text.lower()
    started = False
    for phrase in ("most common adverse reactions", "the most common adverse reactions",
                   "most frequent adverse reactions", "common adverse reactions",
                   "most common side effects", "common side effects"):
        idx = lower.find(phrase)
        if idx != -1:
            text = text[idx:]
            started = True
            break
    if not started and len(text) > 300:
        return None
    parts = re.split(r"(?<!\d)\.(?!\d)", text)
    if parts:
        text = parts[0] + "."
    text = _clean_side_effect_sentence(text)
    if text:
        text = text[0].upper() + text[1:]
    return text


def extract_short_warning(text):
    if not text:
        return None
    text = re.sub(r"\s+", " ", text).strip().replace("BOXED WARNING", "").replace("WARNING:", "").strip()
    lower = text.lower()
    if "lactic acidosis" in lower:
        return "Rare but serious warning: metformin can cause lactic acidosis, especially with kidney problems, heavy alcohol use, or severe illness."
    if "bleeding" in lower:
        return "Important warning: this drug can increase the risk of serious bleeding."
    if "cardiovascular thrombotic events" in lower or "gastrointestinal" in lower:
        return "Important warning: NSAIDs like ibuprofen can increase the risk of stomach bleeding and heart-related events."
    if "suicidal thoughts" in lower or "suicidality" in lower:
        return "Important warning: antidepressants may increase suicidal thoughts in some children, teens, and young adults."
    parts = re.split(r"(?<!\d)\.(?!\d)", text)
    if parts and parts[0].strip():
        return parts[0].strip() + "."
    return text.strip()


def format_side_effect_response(drug_name, label_info):
    """Deterministic fallback text (used if the LLM is off)."""
    if label_info is None:
        fb = COMMON_SIDE_EFFECT_FALLBACKS.get(normalize_key(drug_name))
        if fb:
            return f"{drug_name.title()} side effects:\n- {fb}"
        return f"I could not find clear side-effect information for {drug_name}."
    adverse = extract_short_side_effects(label_info.get("adverse_reactions"), drug_name=drug_name)
    boxed = extract_short_warning(label_info.get("boxed_warning"))
    warnings = extract_short_warning(label_info.get("warnings"))
    out = [f"{drug_name.title()} side effects:"]
    out.append(f"- {adverse}" if adverse else "- I found a label, but it did not list clear common side effects.")
    if boxed:
        out.append(f"- {boxed}")
    elif warnings:
        out.append(f"- {warnings}")
    out.append("\nEducational only -- confirm medication questions with a pharmacist or clinician.")
    return "\n".join(out).strip()


def build_fda_evidence(drug_name):
    """Compact evidence dict for the OpenAI grounding layer."""
    label = get_openfda_label(drug_name)
    return {
        "source": "openFDA drug label",
        "drug": drug_name,
        "common_side_effects": extract_short_side_effects(
            (label or {}).get("adverse_reactions"), drug_name=drug_name),
        "key_warning": extract_short_warning(
            (label or {}).get("boxed_warning")) or extract_short_warning((label or {}).get("warnings")),
    }, label


# Words that mean the user wants side-effect / warning info about a drug.
SIDE_EFFECT_TRIGGERS = (
    "side effect", "side-effect", "side effects", "adverse", "reaction",
    "warning", "warnings", "boxed warning", "what does it do", "tell me about",
    "more info", "more information", "what are the risks of taking",
)


def maybe_fda_answer(text, state):
    """If the user is asking about a single drug's side effects/warnings, answer
    from openFDA (worded by the LLM). Otherwise return (None, state) so the normal
    interaction flow continues untouched."""
    lower = text.lower()
    if not any(t in lower for t in SIDE_EFFECT_TRIGGERS):
        return None, state
    drugs_in_msg = extract_drugs(text)
    # Guard: mid-intake, only answer side-effect questions when a drug is explicitly
    # named, so answers like "no adverse reactions" don't derail the intake flow.
    if not drugs_in_msg and state.get("stage") != "ready":
        return None, state
    drugs = drugs_in_msg or ([state["drugs"][-1]] if state.get("drugs") else [])
    if not drugs:
        return None, state
    drug = drugs[0]
    evidence, _ = build_fda_evidence(drug)
    fallback = format_side_effect_response(drug, get_openfda_label(drug))
    reply = call_openai(
        task=("Explain this single drug's common side effects and most important warning "
              "in a calm, plain, human way. Use ONLY the openFDA evidence provided. "
              "Keep it short and end with a one-line educational disclaimer."),
        state=state,
        evidence=[evidence],
        fallback=fallback,
    )
    return reply, state


print("openFDA side-effect/warning layer ready.")


openFDA side-effect/warning layer ready.


## 8. Chatbot Logic

This is the main controller for the chatbot. The function `handle_user_message()` receives the user's message and the current session state, then decides what should happen next.

The logic is stage-based:

1. `medications`: collect at least two medications.
2. `dosing`: ask for dose or timing information if the user knows it.
3. `allergies`: collect medication allergies.
4. `conditions`: collect important health factors.
5. `age`: collect age group.
6. `concern`: ask what the user wants to know.
7. `confirm`: summarize the profile and ask the user to confirm.
8. `ready`: query DDInter and answer follow-up questions.

This section is the main reason the chatbot feels like a longer conversation instead of a single lookup tool.


In [13]:

def intro_message():
    fallback = (
        "Hi, I am MedChat. I will guide you through a drug interaction check like a short intake interview.\n\n"
        "I will ask about your medications, dose or timing if you know it, allergies, health conditions, age group, "
        "and your main concern. Then I will check the DDInter dataset and explain the recommendation.\n\n"
        "What medications are you currently taking?"
    )
    return call_openai(
        task="Write a friendly opening message explaining the longer intake workflow and asking for medications.",
        state=new_session(),
        fallback=fallback,
    )


def run_recommendation(state):
    state["stage"] = "ready"
    state["confirmed"] = True
    evidence = check_interaction_pairs(state["drugs"])
    state["last_evidence"] = evidence
    if evidence:
        state["last_pair"] = [evidence[-1]["drug1"], evidence[-1]["drug2"]]
    fallback = fallback_recommendation(state, evidence)
    # Enrich the LLM's grounding with openFDA side-effect/warning context (best-effort).
    fda_context = []
    for drug in state["drugs"][:4]:
        try:
            info, _ = build_fda_evidence(drug)
            if info.get("common_side_effects") or info.get("key_warning"):
                fda_context.append(info)
        except Exception:
            pass
    llm_evidence = evidence + fda_context
    return call_openai(
        "Explain DDInter interaction results and recommendations naturally after a full intake "
        "conversation. You may add relevant side effects or warnings, but ONLY from the openFDA "
        "evidence provided.", state, llm_evidence, fallback), state


def handle_user_message(user_text, state):
    state = normalize_session(state)
    state["turns"] += 1
    text = user_text.strip()
    lower = text.lower()

    if not text:
        return next_question(state), state

    if detect_emergency(text):
        response = (
            "This may be an emergency symptom. Please seek urgent medical care now or call emergency services. "
            "For medication poisoning or overdose, contact poison control or emergency care immediately."
        )
        return response, state

    # openFDA side-effect / warning questions (single drug) short-circuit here.
    fda_reply, state = maybe_fda_answer(text, state)
    if fda_reply:
        return fda_reply, state

    if lower in {"reset", "start over", "clear", "new session"}:
        state = new_session()
        return intro_message(), state

    if lower in {"profile", "summary", "what do you know"}:
        return profile_text(state) + "\n\n" + next_question(state), state

    if lower.startswith("remove") or "stop taking" in lower or "no longer take" in lower:
        drugs_to_remove = extract_drugs(text)
        removed = []
        for drug in drugs_to_remove:
            if drug in state["drugs"]:
                state["drugs"].remove(drug)
                removed.append(drug)
        if state.get("last_pair") and any(drug in state["last_pair"] for drug in removed):
            state["last_pair"] = None
        if len(state["drugs"]) < 2:
            state["stage"] = "medications"
        response = f"Removed {', '.join(removed)}." if removed else "I could not identify which medication to remove."
        return response + "\n\n" + next_question(state), state

    mentioned_drugs = extract_drugs(text)
    mentioned_allergies = extract_allergies(text)
    mentioned_conditions = detect_conditions(text)
    dosing_notes = extract_dosing_notes(text)
    age_group = detect_age_group(text)

    if state["stage"] == "medications":
        add_unique(state["drugs"], mentioned_drugs)
        add_unique(state["dosing_notes"], dosing_notes)
        if len(state["drugs"]) >= 2:
            state["stage"] = "dosing"
            fallback = f"I recorded: {', '.join(state['drugs'])}.\n\n{next_question(state)}"
            return call_openai("Acknowledge recorded medications and ask for dose or timing information.", state, fallback=fallback), state
        if len(state["drugs"]) == 1:
            fallback = f"I recorded {state['drugs'][0]}. Please tell me at least one more medication so I can check interactions."
            return call_openai("Acknowledge one medication and ask for another medication.", state, fallback=fallback), state
        return "I could not identify a medication yet. Please type medicine names, for example: `warfarin, ibuprofen`.", state

    if state["stage"] == "dosing":
        add_unique(state["dosing_notes"], dosing_notes)
        state["stage"] = "allergies"
        fallback = "Thanks. " + next_question(state)
        return call_openai("Acknowledge dose or timing details if provided, then ask about allergies.", state, fallback=fallback), state

    if state["stage"] == "allergies":
        if not is_negative_answer(text):
            add_unique(state["allergies"], mentioned_allergies)
        state["stage"] = "conditions"
        fallback = "Thanks. " + next_question(state)
        return call_openai("Acknowledge allergy answer and ask about health conditions.", state, fallback=fallback), state

    if state["stage"] == "conditions":
        if not is_negative_answer(text):
            add_unique(state["conditions"], mentioned_conditions)
        state["stage"] = "age"
        fallback = "Got it. " + next_question(state)
        return call_openai("Acknowledge health factors and ask for age group.", state, fallback=fallback), state

    if state["stage"] == "age":
        if age_group and not is_negative_answer(text):
            state["age_group"] = age_group
        state["stage"] = "concern"
        fallback = "Thanks. " + next_question(state)
        return call_openai("Acknowledge age group and ask for the user's main concern.", state, fallback=fallback), state

    if state["stage"] == "concern":
        if not is_negative_answer(text):
            state["main_concern"] = text
        state["stage"] = "confirm"
        fallback = profile_text(state) + "\n\n" + next_question(state)
        return call_openai("Summarize the intake profile and ask the user to confirm before checking interactions.", state, fallback=fallback), state

    if state["stage"] == "confirm":
        if is_affirmative_answer(text):
            return run_recommendation(state)
        # Treat non-yes confirmation text as corrections or additions.
        add_unique(state["drugs"], mentioned_drugs)
        add_unique(state["dosing_notes"], dosing_notes)
        add_unique(state["allergies"], mentioned_allergies)
        add_unique(state["conditions"], mentioned_conditions)
        if age_group:
            state["age_group"] = age_group
        fallback = profile_text(state) + "\n\nReply `yes` when this is correct, or add another correction."
        return call_openai("Acknowledge corrections and ask for confirmation again.", state, fallback=fallback), state

    # Ready stage: follow-ups and new medication additions.
    new_drugs = [drug for drug in mentioned_drugs if drug not in state["drugs"]]
    if new_drugs:
        add_unique(state["drugs"], new_drugs)
        return run_recommendation(state)

    if any(phrase in lower for phrase in ["what could happen", "what happens", "tell me more", "why", "danger", "risk"]):
        evidence = state.get("last_evidence", [])
        fallback = fallback_recommendation(state, evidence)
        return call_openai("Answer the user's follow-up using the existing DDInter evidence.", state, evidence, fallback), state

    fallback = "I can keep helping. You can ask `what could happen?`, add another medication, remove a medication, or type `profile`."
    return call_openai("Respond helpfully to a general follow-up in this medication interaction session.", state, state.get("last_evidence", []), fallback), state


## 9. Scripted Demo Conversation

This cell runs a fixed conversation without opening the Gradio interface. I included it because it is useful for debugging and for explaining the system during a presentation.

The script shows the full flow: medications, dose/timing, allergies, conditions, age group, user concern, confirmation, recommendation, and follow-up questions.


In [14]:

demo_state = new_session()
print("ASSISTANT:", intro_message())

script = [
    "I take warfarin and ibuprofen",
    "warfarin 5 mg at night and ibuprofen as needed",
    "none",
    "I have kidney disease",
    "adult",
    "I want to know if it is safe and what to watch for",
    "yes",
    "what could happen?",
    "what about aspirin?",
    "profile",
]

for message in script:
    print("\nUSER:", message)
    reply, demo_state = handle_user_message(message, demo_state)
    print("ASSISTANT:", reply)


ASSISTANT: Hello! I’m here to help you check for possible drug interactions. To do this thoroughly, I’ll ask about your medications one at a time, so we can review any potential issues together. Could you please start by telling me the name of one medication or supplement you are currently taking?

USER: I take warfarin and ibuprofen
ASSISTANT: I see you are taking Warfarin and Ibuprofen. Could you please tell me the doses or how often you take each medication? This helps me provide accurate information.

USER: warfarin 5 mg at night and ibuprofen as needed
ASSISTANT: You mentioned Warfarin 5 mg at night and Ibuprofen as needed. Do you have any allergies to medications or other substances I should know about?

USER: none
ASSISTANT: Thank you for sharing that you don't have any allergies. Do you have any current health conditions I should know about?

USER: I have kidney disease
ASSISTANT: Thank you for sharing that you have kidney disease and are taking Warfarin and Ibuprofen. To bette

## 10. Simple Evaluation

This section checks the two most important non-LLM parts of the system:

1. Did the entity extraction find the correct medications?
2. Did the database lookup return the expected severity label?

The test set is small, but it gives a quick sanity check that the core pipeline works. For a larger project, this could be expanded into a bigger evaluation table with more user question styles and more drug pairs.


In [15]:
TEST_CASES = [
    ("I take warfarin and ibuprofen", ["Warfarin", "Ibuprofen"], "Major"),
    ("Can I take Advil with Coumadin?", ["Ibuprofen", "Warfarin"], "Major"),
    ("metformin with omeprazole", ["Metformin", "Omeprazole"], "Unknown"),
    ("tylenol and warfarin", ["Acetaminophen", "Warfarin"], "Moderate"),
]

correct_entities = 0
correct_lookup = 0

for text, expected_drugs, expected_level in TEST_CASES:
    found = extract_drugs(text)
    has_entities = all(drug in found for drug in expected_drugs)
    if has_entities:
        correct_entities += 1
    result = find_interaction(expected_drugs[0], expected_drugs[1])
    level = result["Level"] if result else None
    if level == expected_level:
        correct_lookup += 1
    print(text)
    print("  extracted:", found)
    print("  expected drugs:", expected_drugs, "entity pass:", has_entities)
    print("  lookup level:", level, "expected:", expected_level)

print("\nEntity extraction accuracy:", f"{correct_entities}/{len(TEST_CASES)}")
print("Lookup severity accuracy:", f"{correct_lookup}/{len(TEST_CASES)}")


I take warfarin and ibuprofen
  extracted: ['Warfarin', 'Ibuprofen']
  expected drugs: ['Warfarin', 'Ibuprofen'] entity pass: True
  lookup level: Major expected: Major
Can I take Advil with Coumadin?
  extracted: ['Ibuprofen', 'Warfarin']
  expected drugs: ['Ibuprofen', 'Warfarin'] entity pass: True
  lookup level: Major expected: Major
metformin with omeprazole
  extracted: ['Metformin', 'Omeprazole']
  expected drugs: ['Metformin', 'Omeprazole'] entity pass: True
  lookup level: Unknown expected: Unknown
tylenol and warfarin
  extracted: ['Warfarin', 'Acetaminophen']
  expected drugs: ['Acetaminophen', 'Warfarin'] entity pass: True
  lookup level: Moderate expected: Moderate

Entity extraction accuracy: 4/4
Lookup severity accuracy: 4/4


## 10A. Error Analysis

This section explains where the model and chatbot can make mistakes. The first graph is a confusion matrix for the champion severity classifier, showing which severity classes are most often confused. The second graph summarizes the 50-turn chatbot evaluation checks. These plots are displayed inline in the notebook instead of being exported as image files.

In [16]:
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score
import json

champion_predictions = champion_model.predict(X_test)
labels = sorted(pd.Series(y_test).unique())
cm = confusion_matrix(y_test, champion_predictions, labels=labels)

accuracy = accuracy_score(y_test, champion_predictions)
weighted_f1 = f1_score(y_test, champion_predictions, average="weighted")
macro_f1 = f1_score(y_test, champion_predictions, average="macro")

errors = pd.DataFrame({
    "pair_text": X_test.values,
    "actual": y_test.values,
    "predicted": champion_predictions,
})
errors = errors[errors["actual"] != errors["predicted"]]
error_pairs = (
    errors.groupby(["actual", "predicted"]).size().reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(10)
)

print(f"Champion model: {champion_model_name}")
print(f"Accuracy: {accuracy:.3f}")
print(f"Weighted F1: {weighted_f1:.3f}")
print(f"Macro F1: {macro_f1:.3f}")
print(f"Misclassified test examples: {len(errors):,} out of {len(y_test):,}")
print("\nMost common classification error types:")
display(error_pairs)

chatbot_eval_path = OUTPUT_PATH / "medchat_50_round_eval.json"
if chatbot_eval_path.exists():
    eval_data = json.loads(chatbot_eval_path.read_text())
    eval_summary = eval_data["summary"]
    check_rows = pd.DataFrame([
        {"check": "Chatbot flow", "passed": eval_summary["chatbot_flow_ok"], "total": eval_summary["total_turns"]},
        {"check": "Drug extraction", "passed": eval_summary["entity_ok"], "total": eval_summary["entity_turns"]},
        {"check": "Severity checks", "passed": eval_summary["level_ok"], "total": eval_summary["level_turns"]},
        {"check": "RAG pair retrieval", "passed": eval_summary["rag_ok"], "total": eval_summary["rag_pair_turns"]},
    ])
else:
    check_rows = pd.DataFrame([
        {"check": "Chatbot flow", "passed": 0, "total": 0},
        {"check": "Drug extraction", "passed": 0, "total": 0},
        {"check": "Severity checks", "passed": 0, "total": 0},
        {"check": "RAG pair retrieval", "passed": 0, "total": 0},
    ])

check_rows["failed"] = check_rows["total"] - check_rows["passed"]
print("\nChatbot/RAG evaluation checks:")
display(check_rows)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title(f"{champion_model_name} Confusion Matrix")
axes[0].set_xlabel("Predicted severity")
axes[0].set_ylabel("Actual severity")
check_rows.set_index("check")[["passed", "failed"]].plot(
    kind="bar", stacked=True, ax=axes[1], color=["#2E7D32", "#C62828"]
)
axes[1].set_title("Chatbot Evaluation Pass/Fail")
axes[1].set_xlabel("Evaluation check")
axes[1].set_ylabel("Number of checks")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

print("Error analysis notes:")
print("- Exact DDInter lookup is the most reliable path when a drug pair exists.")
print("- ML predictions can confuse nearby severity classes, especially when the drug-pair text is sparse.")
print("- Brand names or rare misspellings may fail if they are not covered by synonym or fuzzy matching.")
print("- RAG retrieval can return similar context, so the chatbot should still prefer exact DDInter evidence.")
print("- LLM responses are grounded with evidence, but final medical decisions should be confirmed by a clinician.")

Champion model: Linear SVM
Accuracy: 0.815
Weighted F1: 0.819
Macro F1: 0.746
Misclassified test examples: 2,216 out of 12,000

Most common classification error types:


,actual,predicted,count
6,Moderate,Major,634
8,Moderate,Unknown,506
1,Major,Moderate,354
7,Moderate,Minor,247
4,Minor,Moderate,110
11,Unknown,Moderate,102
2,Major,Unknown,61
0,Major,Minor,56
10,Unknown,Minor,44
5,Minor,Unknown,41



Chatbot/RAG evaluation checks:


,check,passed,total,failed
0,Chatbot flow,0,0,0
1,Drug extraction,0,0,0
2,Severity checks,0,0,0
3,RAG pair retrieval,0,0,0


Error analysis notes:
- Exact DDInter lookup is the most reliable path when a drug pair exists.
- ML predictions can confuse nearby severity classes, especially when the drug-pair text is sparse.
- Brand names or rare misspellings may fail if they are not covered by synonym or fuzzy matching.
- RAG retrieval can return similar context, so the chatbot should still prefer exact DDInter evidence.
- LLM responses are grounded with evidence, but final medical decisions should be confirmed by a clinician.


## 11. Branded MedChat Web UI (Gradio)

The live demo: a MedChat-branded interface with the guided intake chat, a live medications panel, severity legend, an openFDA-aware "How It Works" panel, and example prompts. `share=True` gives a public link you can open on Colab.


In [17]:
# ============================================================
# Branded MedChat UI (Gradio) wired to the guided chatbot backend.
# ============================================================
import gradio as gr

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800;900&display=swap');

* { font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important; box-sizing: border-box !important; }

html, body {
    background:
        radial-gradient(circle at 6% 10%, rgba(129, 140, 248, 0.16), transparent 27%),
        radial-gradient(circle at 92% 10%, rgba(192, 132, 252, 0.13), transparent 24%),
        linear-gradient(180deg, #eef2ff 0%, #f8fafc 100%) !important;
}
.gradio-container { background: transparent !important; min-height: 100vh !important; max-width: 100% !important; padding: 0 !important; }
footer { display: none !important; }

.mc-header {
    width: min(1500px, calc(100% - 48px)); margin: 22px auto 0;
    background:
        radial-gradient(circle at 15% 15%, rgba(168, 85, 247, 0.31), transparent 28%),
        radial-gradient(circle at 82% 22%, rgba(99, 102, 241, 0.38), transparent 30%),
        linear-gradient(135deg, #1e1b4b 0%, #312e81 42%, #4338ca 100%);
    border-radius: 34px; padding: 34px 44px; display: flex; align-items: center;
    justify-content: space-between; gap: 30px; box-shadow: 0 20px 56px rgba(30, 27, 75, 0.33);
    position: relative; overflow: hidden;
}
.mc-header::before { content: ""; position: absolute; width: 260px; height: 260px; right: -78px; top: -95px; background: rgba(255,255,255,0.09); border-radius: 999px; }
.mc-header::after  { content: ""; position: absolute; width: 170px; height: 170px; left: 40%; bottom: -115px; background: rgba(255,255,255,0.07); border-radius: 999px; }
.mc-logo-wrap { display: flex; align-items: center; gap: 22px; position: relative; z-index: 2; min-width: 0; flex: 1 1 auto; }
.mc-logo-icon { width: 86px; height: 86px; border-radius: 27px; background: rgba(255,255,255,0.15); display: flex; align-items: center; justify-content: center; font-size: 44px; box-shadow: inset 0 0 0 1px rgba(255,255,255,0.22); backdrop-filter: blur(12px); flex: 0 0 86px; }
.mc-logo-name { font-size: 3.0em; line-height: 0.95; font-weight: 900; letter-spacing: -0.05em; color: #ffffff; white-space: nowrap; }
.mc-logo-tagline { margin-top: 8px; color: #ddd6fe; font-size: 1.0em; font-weight: 800; letter-spacing: 0.07em; text-transform: uppercase; white-space: nowrap; }
.mc-header-sub { margin-top: 8px; color: #c7d2fe; font-size: 1.0em; font-weight: 500; white-space: nowrap; }
.mc-pills-wrap { position: relative; z-index: 2; display: flex; flex-direction: column; align-items: flex-end; gap: 12px; flex: 0 0 auto; min-width: 0; }
.mc-pills-title { color: #c7d2fe; font-size: 0.72em; font-weight: 900; letter-spacing: 0.18em; text-transform: uppercase; white-space: nowrap; }
.mc-pills { display: flex; gap: 8px; flex-wrap: nowrap; justify-content: flex-end; max-width: none; overflow-x: auto; scrollbar-width: none; }
.mc-pills::-webkit-scrollbar { display: none; }
.mc-pill { color: white; background: rgba(255,255,255,0.14); border: 1px solid rgba(255,255,255,0.20); padding: 8px 11px; border-radius: 999px; font-size: 0.78em; font-weight: 800; box-shadow: 0 7px 22px rgba(0,0,0,0.08); white-space: nowrap; flex: 0 0 auto; }
.mc-disclaimer { width: min(1500px, calc(100% - 48px)); margin: 14px auto 0; background: #fff7ed; border: 1px solid #fed7aa; color: #9a3412; border-radius: 18px; padding: 12px 18px; font-size: 0.95em; font-weight: 700; text-align: center; }
.mc-page { width: min(1500px, calc(100% - 48px)); margin: 18px auto 28px; }
.mc-main-row { gap: 20px !important; align-items: stretch !important; }
.mc-chat-col, .mc-side-col { background: rgba(255,255,255,0.83); border: 1px solid rgba(199,210,254,0.85); border-radius: 30px; padding: 18px !important; box-shadow: 0 18px 55px rgba(79,70,229,0.13); backdrop-filter: blur(16px); }
.chat-window { border-radius: 22px !important; overflow: hidden !important; background: #ffffff !important; border: 1px solid #e0e7ff !important; }
.mc-input-row { margin-top: 14px !important; gap: 12px !important; align-items: stretch !important; }
.mc-input textarea { border-radius: 18px !important; border: 1px solid #c7d2fe !important; min-height: 54px !important; font-size: 1.02em !important; padding: 14px 16px !important; background: #ffffff !important; }
.mc-send-btn button { min-height: 54px !important; border-radius: 18px !important; border: 0 !important; background: linear-gradient(135deg, #4f46e5 0%, #7c3aed 100%) !important; color: white !important; font-weight: 900 !important; font-size: 1.01em !important; box-shadow: 0 13px 28px rgba(79,70,229,0.25) !important; }
.mc-card { background: #ffffff; border: 1px solid #e0e7ff; border-radius: 24px; padding: 18px; margin-bottom: 15px; box-shadow: 0 10px 26px rgba(79,70,229,0.08); }
.mc-card-title { display: block; font-size: 0.78em; font-weight: 900; color: #4338ca; letter-spacing: 0.14em; text-transform: uppercase; margin-bottom: 13px; }
.mc-meds-box textarea { background: #f8fafc !important; border: 1px solid #e0e7ff !important; border-radius: 16px !important; color: #1e1b4b !important; font-weight: 700 !important; line-height: 1.55 !important; }
.mc-clear-btn button { border-radius: 15px !important; border: 1px solid #e0e7ff !important; color: #4338ca !important; font-weight: 800 !important; background: #f8fafc !important; }
.sev-row { display: flex; align-items: center; gap: 10px; margin: 9px 0; }
.badge { min-width: 86px; text-align: center; padding: 7px 9px; border-radius: 999px; color: white; font-weight: 900; font-size: 0.75em; }
.b-major { background: #dc2626; } .b-mod { background: #f59e0b; } .b-minor { background: #16a34a; } .b-unknown { background: #64748b; }
.sev-desc { color: #475569; font-size: 0.88em; font-weight: 650; }
.mc-step { display: flex; gap: 11px; margin: 11px 0; align-items: flex-start; }
.mc-step-num { width: 26px; height: 26px; border-radius: 999px; background: #eef2ff; color: #4f46e5; display: flex; align-items: center; justify-content: center; font-size: 0.82em; font-weight: 900; flex: 0 0 26px; }
.mc-step-text { color: #475569; font-size: 0.91em; line-height: 1.35; }
.mc-step-bold { color: #1e1b4b; font-weight: 900; }
.mc-ex { display: flex; gap: 9px; padding: 9px 0; border-bottom: 1px solid #eef2ff; }
.mc-ex:last-child { border-bottom: 0; }
.mc-ex-dot { color: #7c3aed; font-weight: 900; }
.mc-ex-text { color: #334155; font-size: 0.91em; font-weight: 650; }
.mc-example-btn button { width: 100% !important; border-radius: 14px !important; background: #f8fafc !important; color: #4338ca !important; border: 1px solid #e0e7ff !important; font-weight: 800 !important; }
@media (max-width: 1050px) {
    .mc-header, .mc-disclaimer, .mc-page { width: calc(100% - 24px); }
    .mc-header { padding: 28px 24px; flex-direction: column; align-items: flex-start; }
    .mc-logo-name { font-size: 2.45em; }
    .mc-logo-tagline, .mc-header-sub { white-space: normal; }
    .mc-pills-wrap { align-items: flex-start; width: 100%; }
    .mc-pills { justify-content: flex-start; width: 100%; }
    .mc-main-row { flex-direction: column !important; }
}
"""


def format_drugs_display(state):
    drugs = (state or {}).get("drugs", [])
    if drugs:
        return "Current session medications:\n" + "\n".join(f"\u2022 {d}" for d in drugs)
    return "No medications tracked yet"


def _content_to_text(content):
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return _content_to_text(content.get("text", content.get("content", "")))
    if isinstance(content, list):
        return "\n".join(t for t in (_content_to_text(i) for i in content) if t)
    return str(content)


def normalize_history(history):
    cleaned = []
    for item in list(history or []):
        if isinstance(item, dict) and item.get("role") in {"user", "assistant"}:
            cleaned.append({"role": item["role"], "content": _content_to_text(item.get("content", ""))})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            u, a = item
            if u is not None:
                cleaned.append({"role": "user", "content": _content_to_text(u)})
            if a is not None:
                cleaned.append({"role": "assistant", "content": _content_to_text(a)})
    return cleaned


def chat(user_message, history, session_state):
    session_state = normalize_session(session_state)
    history = normalize_history(history)
    if not user_message or not str(user_message).strip():
        return history, session_state, format_drugs_display(session_state)
    reply, session_state = handle_user_message(str(user_message).strip(), session_state)
    history = history + [
        {"role": "user", "content": str(user_message).strip()},
        {"role": "assistant", "content": str(reply)},
    ]
    return history, session_state, format_drugs_display(session_state)


def reset_session():
    state = new_session()
    history = [{"role": "assistant", "content": intro_message()}]
    return history, state, format_drugs_display(state)


initial_state = new_session()
initial_history = [{"role": "assistant", "content": intro_message()}]

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:
    gr.HTML("""
    <div class="mc-header">
        <div class="mc-logo-wrap">
            <div class="mc-logo-icon">💊</div>
            <div>
                <div class="mc-logo-name">MedChat</div>
                <div class="mc-logo-tagline">Drug Interaction Assistant</div>
                <div class="mc-header-sub">A grounded DDInter + RAG + openFDA medication safety assistant</div>
            </div>
        </div>
        <div class="mc-pills-wrap">
            <div class="mc-pills-title">Powered by</div>
            <div class="mc-pills">
                <span class="mc-pill">RapidFuzz</span>
                <span class="mc-pill">SQLite</span>
                <span class="mc-pill">Linear SVM</span>
                <span class="mc-pill">RAG</span>
                <span class="mc-pill">openFDA</span>
                <span class="mc-pill">OpenAI</span>
                <span class="mc-pill">Gradio</span>
            </div>
        </div>
    </div>
    <div class="mc-disclaimer">
        ⚠&nbsp; Educational/demo project only &mdash; not medical advice. Always consult a healthcare professional before medication decisions.
    </div>
""")

    with gr.Column(elem_classes=["mc-page"]):
        with gr.Row(elem_classes=["mc-main-row"]):
            with gr.Column(scale=8, elem_classes=["mc-chat-col"]):
                # `type="messages"` is required on Gradio 4.44-5.x but was removed in 6.x
                # (messages is the only format there), so add it only when supported.
                _chat_kwargs = dict(value=initial_history, height=590,
                                    show_label=False, elem_classes=["chat-window"])
                try:
                    chatbot = gr.Chatbot(type="messages", **_chat_kwargs)
                except TypeError:
                    chatbot = gr.Chatbot(**_chat_kwargs)
                with gr.Row(elem_classes=["mc-input-row"]):
                    msg = gr.Textbox(placeholder="Type a medication, brand name, condition, or question...",
                                     show_label=False, scale=8, lines=1, max_lines=3,
                                     autofocus=True, elem_classes=["mc-input"])
                    send_btn = gr.Button("Send \u2192", scale=1, min_width=120, elem_classes=["mc-send-btn"])
                with gr.Row():
                    example_1 = gr.Button("Warfarin + ibuprofen", elem_classes=["mc-example-btn"])
                    example_2 = gr.Button("Advil with Coumadin", elem_classes=["mc-example-btn"])
                    example_3 = gr.Button("Metformin side effects", elem_classes=["mc-example-btn"])
                    example_4 = gr.Button("What does Major mean?", elem_classes=["mc-example-btn"])

            with gr.Column(scale=3, elem_classes=["mc-side-col"]):
                gr.HTML('<div class="mc-card"><span class="mc-card-title">Current Medications</span>')
                drugs_display = gr.Textbox(show_label=False, value="No medications tracked yet",
                                           interactive=False, lines=5, max_lines=10, elem_classes=["mc-meds-box"])
                reset_btn = gr.Button("\U0001F5D1  Clear Session", elem_classes=["mc-clear-btn"], variant="secondary")
                gr.HTML("</div>")
                gr.HTML("""
                <div class="mc-card">
                    <span class="mc-card-title">Severity Levels</span>
                    <div class="sev-row"><span class="badge b-major">Major</span><span class="sev-desc">Avoid / get professional advice</span></div>
                    <div class="sev-row"><span class="badge b-mod">Moderate</span><span class="sev-desc">Caution / monitoring may be needed</span></div>
                    <div class="sev-row"><span class="badge b-minor">Minor</span><span class="sev-desc">Lower concern but still relevant</span></div>
                    <div class="sev-row"><span class="badge b-unknown">Unknown</span><span class="sev-desc">Insufficient database evidence</span></div>
                </div>
""")
                gr.HTML("""
                <div class="mc-card">
                    <span class="mc-card-title">How It Works</span>
                    <div class="mc-step"><div class="mc-step-num">1</div><div class="mc-step-text"><span class="mc-step-bold">Drug extraction</span> finds drug names, brands, and typos (RapidFuzz)</div></div>
                    <div class="mc-step"><div class="mc-step-num">2</div><div class="mc-step-text"><span class="mc-step-bold">DDInter lookup</span> checks the exact pair in SQLite</div></div>
                    <div class="mc-step"><div class="mc-step-num">3</div><div class="mc-step-text"><span class="mc-step-bold">Linear SVM</span> predicts severity when no exact pair exists</div></div>
                    <div class="mc-step"><div class="mc-step-num">4</div><div class="mc-step-text"><span class="mc-step-bold">RAG</span> retrieves similar DDInter records for grounding</div></div>
                    <div class="mc-step"><div class="mc-step-num">5</div><div class="mc-step-text"><span class="mc-step-bold">openFDA</span> adds side-effect and warning summaries</div></div>
                    <div class="mc-step"><div class="mc-step-num">6</div><div class="mc-step-text"><span class="mc-step-bold">OpenAI</span> explains the evidence in natural language</div></div>
                </div>
""")
                gr.HTML("""
                <div class="mc-card">
                    <span class="mc-card-title">Try Asking</span>
                    <div class="mc-ex"><span class="mc-ex-dot">›</span><span class="mc-ex-text">I take warfarin and ibuprofen</span></div>
                    <div class="mc-ex"><span class="mc-ex-dot">›</span><span class="mc-ex-text">Can I take Advil with Coumadin?</span></div>
                    <div class="mc-ex"><span class="mc-ex-dot">›</span><span class="mc-ex-text">What are the side effects of metformin?</span></div>
                    <div class="mc-ex"><span class="mc-ex-dot">›</span><span class="mc-ex-text">What could happen if I combine them?</span></div>
                    <div class="mc-ex"><span class="mc-ex-dot">›</span><span class="mc-ex-text">What does a Major interaction mean?</span></div>
                </div>
""")

    session_state = gr.State(initial_state)

    send_btn.click(chat, [msg, chatbot, session_state], [chatbot, session_state, drugs_display]).then(lambda: "", None, [msg])
    msg.submit(chat, [msg, chatbot, session_state], [chatbot, session_state, drugs_display]).then(lambda: "", None, [msg])
    reset_btn.click(reset_session, None, [chatbot, session_state, drugs_display])
    example_1.click(lambda: "I take warfarin and ibuprofen", None, [msg])
    example_2.click(lambda: "Can I take Advil with Coumadin?", None, [msg])
    example_3.click(lambda: "What are the side effects of metformin?", None, [msg])
    example_4.click(lambda: "What does a Major interaction mean?", None, [msg])

print("Gradio version:", gr.__version__)
demo.launch(share=True, debug=False)


Gradio version: 6.20.0
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://59bd922cd3b06dc22d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 12. Project Summary and Notes

This project combines several topics from the course/project instructions:

- information extraction: medication names are extracted from user text
- question answering: the user asks medication safety questions
- intelligent chatbot: the assistant follows a multi-turn conversation
- database use: DDInter is stored and queried with SQLite
- classification: multiple ML models are compared, and the champion model predicts severity classes for missing pairs
- evaluation: extraction and lookup are tested with sample cases

The strongest part of the project is that the LLM is not used as the medical database. The dataset lookup and champion ML model happen first, and the LLM only explains the retrieved or predicted evidence in a more natural way.

Limitations:

- The DDInter coverage depends on the CSV files available in this folder.
- The chatbot does not know dosage-specific clinical rules.
- The safety recommendation is educational and should not replace a pharmacist or doctor.
- The evaluation set is small and should be expanded for a more complete study.

Possible future improvements:

- Add RxNorm or another standard drug-normalization system.
- Add openFDA information for side effects and warnings.
- Add more test cases and calculate precision/recall for entity extraction.
- Add a larger recommender or prediction model for missing drug pairs.
